# ARoG on Synthea: Protocol A, test run

This notebook runs the pre-registered test of Protocol A once. The order is fixed: set up the environment, freeze the pre-registration (`PREREG_LOCK.json`), build the test data, run ARoG in three chunks, then analyse.
Code and parameters are not changed after the FREEZE cell. If a cell fails, the error is recorded and the run is not tuned to produce a result.

Requirements: the `aciids-fv` Kaggle dataset version that contains `aciids_fv/` and the dev outputs in `aciids_fv/out/dev_protocolA/`; GPU T4 x1; internet on.

Test sample (fixed before the run): the first 20 patients per class by patient ID, 120 patients x {L0, L1, L2} = 360 ARoG records (about 3.1 h at the 31 s/record measured on dev).

Primary endpoint: accuracy of A3_logreg on the relation-label set parsed from the real ARoG prompts at L0 and L1; one-sided exact binomial test against 1/6 with Holm correction (see `PREREG_LOCK.json`).

In [ ]:
# Paths and run flags (fixed before FREEZE)
WORK = "/kaggle/working"
PREREG = f"{WORK}/PREREG_LOCK.json"
OLLAMA_TAG = "qwen2.5:7b-instruct"
N_PATIENTS = 120          # fixed in the pre-registration (20 per class)
LEVELS = "L0,L1,L2"
RUN_ARoG_CHUNKS = [0, 1, 2]   # after a session restart, keep only unfinished chunks
RUN_A1_RAW   = True      # A1 on raw transcripts (~1 h)
RUN_ABLATION = True      # A1 stage ablation
RUN_A1_FIXED = True      # A1 on fixed views, same patients (~35 min)
print("OK")

## 1. Environment: Ollama, ARoG, Virtuoso, lm_server

In [ ]:
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)

!ollama pull qwen2.5:7b-instruct

In [ ]:
import requests, time

# server is up
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
models = [m["name"] for m in r.json()["models"]]
print("Models:", models)
assert any("qwen2.5:7b-instruct" in m for m in models), "model not pulled"

# model answers (first call loads it onto the GPU, 10-30 s)
t = time.time()
r = requests.post("http://127.0.0.1:11434/api/chat", json={
    "model": "qwen2.5:7b-instruct",
    "messages": [{"role": "user", "content": "Answer with one word: 2+2=?"}],
    "stream": False,
    "options": {"temperature": 0, "num_predict": 20},
}, timeout=180)
print("Reply:", r.json()["message"]["content"], f"({time.time()-t:.1f}s)")
print("OK")

In [ ]:
!ollama ps
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
!git clone https://github.com/NLPGM/ARoG.git
%cd ARoG

In [ ]:
!apt-get update
!apt-get install python3.10 python3.10-venv python3.10-dev -y
!python3.10 -m venv arog_env

In [ ]:
!arog_env/bin/pip install -r requirements.txt

In [ ]:
!arog_env/bin/pip uninstall torch torchvision torchaudio -y
!arog_env/bin/pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!arog_env/bin/python -c "import torch; print('CUDA available:', torch.cuda.is_available())"

In [ ]:
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq virtuoso-opensource > /dev/null 2>&1
!which virtuoso-t isql-vt

In [ ]:
import os, re, time, subprocess

os.system("pkill -9 virtuoso-t")
os.system("find / -name virtuoso.lck -delete 2>/dev/null")

ini_path = os.popen("find /etc /var -name virtuoso.ini 2>/dev/null | head -n 1").read().strip()
print("ini:", ini_path)

# allow Virtuoso to read files under /kaggle/working
txt = open(ini_path).read()
if "/kaggle/working" not in txt:
    txt = re.sub(r"(?m)^(DirsAllowed\s*=.*)$", r"\1, /kaggle/working", txt)
    open(ini_path, "w").write(txt)

subprocess.Popen(f"nohup virtuoso-t +configfile {ini_path} +foreground > virtuoso_run.log 2>&1 &", shell=True)

# wait for port 1111
for _ in range(60):
    if os.system("ss -ltn | grep -q ':1111 '") == 0:
        break
    time.sleep(1)
print("Virtuoso:", os.popen("pgrep -l virtuoso-t").read().strip())

In [ ]:
!pip install SPARQLWrapper

In [ ]:
%cd /kaggle/working/ARoG
!sed -i 's#^SPARQLPATH = .*#SPARQLPATH = "http://127.0.0.1:8890/sparql"#' freebase_func.py
!sed -i "s#^SLMPATH = .*#SLMPATH = 'http://127.0.0.1:5001/sentence_transformer'#" freebase_func.py
!grep -n "^SPARQLPATH\|^SLMPATH" freebase_func.py

In [ ]:
import os, subprocess, time, requests

# run the sentence encoder on CUDA (idempotent)
code = open("lm_server.py", encoding="utf-8").read()
if "device='cuda'" not in code:
    code = code.replace(
        "SentenceTransformer('sentence-transformers/msmarco-distilbert-base-tas-b')",
        "SentenceTransformer('sentence-transformers/msmarco-distilbert-base-tas-b', device='cuda')")
    open("lm_server.py", "w", encoding="utf-8").write(code)

os.system("pkill -9 -f lm_server.py")
time.sleep(1)
subprocess.Popen("nohup ./arog_env/bin/python lm_server.py > lm_server.log 2>&1 &", shell=True)

# wait for the server; the first start downloads the model (1-2 min)
ok = False
for _ in range(180):
    try:
        r = requests.post("http://127.0.0.1:5001/sentence_transformer",
                          json={'width': '2', 'question': 'test',
                                'total_relations': 'a.b.c&;& d.e.f&;& g.h.i'}, timeout=10)
        if r.status_code == 200:
            print("lm_server ready:", r.json()); ok = True; break
    except Exception:
        pass
    time.sleep(1)
if not ok:
    print(open("lm_server.log").read()[-2000:])

## 2. Copy data and freeze the pre-registration

After the FREEZE cell, code and configuration stay unchanged. Scripts refuse to read the test split unless every hash in the lock file matches.

In [ ]:
import glob, os, shutil
from pathlib import Path

def find(name):
    matches = [
        path for path in Path("/kaggle/input").rglob(f"upload/{name}")
        if path.is_dir()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"expected exactly one upload/{name}, found {len(matches)}: {matches}"
        )
    return matches[0]
for name in ("aciids_fv", "ACIIDS_2027"):
    dst = f"{WORK}/{name}"
    if not os.path.exists(dst):
        shutil.copytree(find(name), dst)
need = ["ACIIDS_T01/data_private/raw/kg_v0.1/kg/nodes.csv","ACIIDS_T01/data_private/raw/kg_v0.1/kg/triples.csv",
 "ACIIDS_T01/data_private/raw/kg_v0.1/data/clean/patients_minimal_PRIVATE.csv","ACIIDS_T01/data_private/raw/kg_v0.1/configs/id_key_PRIVATE.hex",
 "ACIIDS_T05_v2_1/data_private/relation_mapping_v2_1_CANDIDATE_PRIVATE.csv","ACIIDS_T02/configs/split_manifest.csv","ACIIDS_T04/configs/banned_tokens.txt"]
missing = [p for p in need if not os.path.exists(f"{WORK}/ACIIDS_2027/{p}")]
assert not missing, f"missing files: {missing}"
assert os.path.exists(f"{WORK}/aciids_fv/protoA/stat_on_transcripts.py"), "outdated dataset version"
assert "perm" in open(f"{WORK}/aciids_fv/protoA/stat_on_transcripts.py").read(), "outdated stat_on_transcripts.py"
print("data and code present")

In [ ]:
# FREEZE: hash code, config, mapping, split and dev outputs into PREREG_LOCK.json
import glob, os
os.chdir(f"{WORK}/aciids_fv")
dev = sorted(glob.glob("out/dev_protocolA/*"))
assert dev, "aciids_fv/out/dev_protocolA/ (dev outputs) missing from the dataset"
assert not os.path.exists(PREREG), "PREREG_LOCK.json already exists; the run is already frozen"
DEVF = " ".join(dev)
!python make_prereg.py --base /kaggle/working/ACIIDS_2027 --attacker_models $OLLAMA_TAG --dev_report $DEVF --out $PREREG
import hashlib
print("SHA-256 of PREREG_LOCK.json:", hashlib.sha256(open(PREREG,"rb").read()).hexdigest())
print("Record this hash and download PREREG_LOCK.json now; it timestamps the freeze.")

In [ ]:
print(open("/kaggle/working/PREREG_LOCK.json").read())

In [ ]:
import json
d = json.load(open("/kaggle/working/PREREG_LOCK.json"))
for k in d["artifact_sha256"]: print(k)
print("system_prompt_sha256:", d["system_prompt_sha256"][:16])

## 3. Test data, Virtuoso load, logging patch

In [ ]:
%cd /kaggle/working/aciids_fv/protoA
!python make_arog_data.py --base /kaggle/working/ACIIDS_2027 --split test --levels $LEVELS --limit_patients $N_PATIENTS --prereg $PREREG --outdir /kaggle/working/testA
# expected: records=360 patients=120 ... lint=PASS

In [ ]:
%cd /kaggle/working/ARoG
!mkdir -p Freebase
!cp /kaggle/working/testA/FilterFreebase Freebase/FilterFreebase
!wc -l Freebase/FilterFreebase

In [ ]:
import os
data_dir = os.path.abspath("Freebase")

sql = f"""
DB.DBA.RDF_GLOBAL_RESET();
delete from DB.DBA.load_list;
ld_dir('{data_dir}', 'FilterFreebase', 'http://freebase.com');
rdf_loader_run();
checkpoint;
"""
open("load_data.sql", "w").write(sql)

print(os.popen("isql-vt 1111 dba dba load_data.sql").read())

In [ ]:
check = """
SELECT ll_state, COUNT(*) FROM DB.DBA.load_list GROUP BY ll_state;
SPARQL SELECT COUNT(*) WHERE { ?s ?p ?o };
"""
open("check.sql", "w").write(check)
print(os.popen("isql-vt 1111 dba dba check.sql").read())

In [ ]:
import json
from SPARQLWrapper import SPARQLWrapper, JSON
qs = json.load(open("/kaggle/working/testA/WebQSP.json"))
subj = list(qs[0]["topic_entity"])[0]
s = SPARQLWrapper("http://127.0.0.1:8890/sparql")
s.setQuery("PREFIX ns: <http://rdf.freebase.com/ns/> SELECT ?r ?x WHERE { ns:%s ?r ?x } LIMIT 5" % subj)
s.setReturnFormat(JSON)
rows = s.query().convert()["results"]["bindings"]
print(subj, "->", [(r["r"]["value"].split("/")[-1], r["x"]["value"].split("/")[-1]) for r in rows])
assert len(rows) > 0, "test data not loaded into Virtuoso"

In [ ]:
%%writefile utils.py
import ast
import random
import re
import time
import json
import logging
import requests

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
from freebase_func import *
import hashlib
import base64

SLMPATH = 'http://127.0.0.1:5001/sentence_transformer'
SLMPATH_COS = 'http://127.0.0.1:5001/sentence_transformer_cos'
SLMPATH_COS_LIST = 'http://127.0.0.1:5001/sentence_transformer_cos_list'

# Ollama backend
OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
OLLAMA_MODEL = "qwen2.5:7b-instruct"


def jsonl_to_json(jsonl_file, json_file):
    with open(jsonl_file, 'r') as infile:
        with open(json_file, 'w') as outfile:
            json_lines = infile.readlines()
            json_list = [json.loads(line) for line in json_lines]
            json.dump(json_list, outfile, indent=4)


def run_llm(prompt, temperature, max_tokens, opeani_api_keys, engine="gpt-3.5-turbo"):
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max(max_tokens, 1024),
            "num_ctx": 8192,
        },
    }
    result = ""
    token_num = {"total": 0, "input": 0, "output": 0}
    for attempt in range(3):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=300)
            response.raise_for_status()
            data = response.json()
            result = data["message"]["content"]
            # strip reasoning blocks if the model emits them
            result = re.sub(r"<think>.*?</think>", "", result, flags=re.DOTALL).strip()
            p = data.get("prompt_eval_count", 0)
            o = data.get("eval_count", 0)
            token_num = {"total": p + o, "input": p, "output": o}
            break
        except Exception as e:
            print("Ollama call failed:", e)
            time.sleep(3)

    logging.info(result)
    logging.info(token_num)
    return result, token_num


def generate_without_explored_paths(question, args):
    prompt = generate_directly + "\n\nQ: " + question + "\nA:"
    generator_response_directly, token_num = run_llm(prompt, args.temperature_reasoning, args.max_length,
                                                     args.opeani_api_keys, args.LLM_type)
    return generator_response_directly, token_num


def prepare_dataset(dataset_name):
    if dataset_name == 'cwq':
        with open('data/cwq_1000.json', encoding='utf-8') as f:
            datas = json.load(f)
        question_string = 'question'
    elif dataset_name == 'webqsp':
        with open('data/WebQSP.json', encoding='utf-8') as f:
            datas = json.load(f)
        question_string = 'RawQuestion'
    elif dataset_name == 'grailqa':
        with open('data/grailqa.json', encoding='utf-8') as f:
            datas = json.load(f)
        question_string = 'question'
    else:
        print("dataset not found, you should pick from {cwq, webqsp, grailqa}.")
        exit(-1)
    return datas, question_string


def generate_abs_question(question, args):
    prompt = question_abstractly + "\n\nQ: " + question
    abs_question_result, token_num = run_llm(prompt, args.temperature_reasoning, args.max_length, args.opeani_api_keys,
                                             args.LLM_type)
    lines = abs_question_result.split('\n')
    thought = ''
    reasoning_path = ''
    answer = ''
    answer_type = ''
    crucial_entities = []
    for line in lines:
        if line.startswith('Thought:'):
            thought = line[len('Thought:'):].strip()
        elif line.startswith('Reasoning Path:'):
            reasoning_path = line[len('Reasoning Path:'):].strip()
        elif line.startswith('Answer:'):
            answer = line[len('Answer:'):].strip()
    pattern = r'\(.*?\)'
    matches = re.findall(pattern, reasoning_path)
    matches = [item.replace('(', '').replace(')', '').strip() for item in matches if item not in answer]
    crucial_entities = list(set(matches))
    answer_type = re.findall(pattern, answer)
    if len(answer_type) > 0:
        answer_type = answer_type[0].replace('(', '').replace(')', '').strip()
    else:
        answer_type = ""
    return thought, reasoning_path, answer, answer_type, crucial_entities, token_num


def simple_hash_encrypt(input_string):
    sha_signature = hashlib.sha256(input_string.encode()).digest()
    encoded_hash = base64.urlsafe_b64encode(sha_signature).decode().rstrip('=\n')
    encoded_hash = encoded_hash.replace('-', '0').replace('_', '1').lower()
    return 'm.' + encoded_hash[:10]


def relation_search_prune(entity_id, entity_name, pre_relations, pre_head, question, args):
    sparql_relations_extract_head = sparql_head_relations % entity_id
    head_relations = execurte_sparql(sparql_relations_extract_head)
    head_relations = replace_relation_prefix(head_relations)

    sparql_relations_extract_tail = sparql_tail_relations % entity_id
    tail_relations = execurte_sparql(sparql_relations_extract_tail)
    tail_relations = replace_relation_prefix(tail_relations)

    head_relations = list(set(head_relations))
    tail_relations = list(set(tail_relations))

    if args.remove_unnecessary_rel:
        head_relations = [relation for relation in head_relations if not abandon_rels(relation)]
        tail_relations = [relation for relation in tail_relations if not abandon_rels(relation)]

    if len(pre_relations) != 0:
        tail_relations = [rel for rel in tail_relations if rel not in pre_relations]
        head_relations = [rel for rel in head_relations if rel not in pre_relations]

    total_relations = head_relations + tail_relations
    total_relations = list(set(total_relations))

    token_num = {'total': 0, 'input': 0, 'output': 0}
    if args.prune_tools == "llm":
        prompt = construct_relation_prune_prompt(question, entity_name, total_relations, args)
        prune_relations_result, token_num = run_llm(prompt, args.temperature_exploration, args.max_length,
                                                    args.opeani_api_keys,
                                                    args.LLM_type)
        flag, retrieve_relations_with_scores = clean_relations(prune_relations_result, entity_id, head_relations,
                                                               total_relations)
    elif args.prune_tools == "bm25":
        topn_relations, topn_scores = compute_bm25_similarity(question, total_relations, args.width)
        flag, retrieve_relations_with_scores = clean_relations_bm25_sent(topn_relations, topn_scores, entity_id,
                                                                         head_relations)
    else:
        model = SentenceTransformer('sentence-transformers/msmarco-distilbert-base-tas-b')
        topn_relations, topn_scores = retrieve_top_docs(question, total_relations, model, args.width)
        flag, retrieve_relations_with_scores = clean_relations_bm25_sent(topn_relations, topn_scores, entity_id,
                                                                         head_relations)

    if flag:
        return retrieve_relations_with_scores, token_num
    else:
        return [], token_num


def construct_relation_prune_prompt(question, entity_name, total_relations, args):
    return extract_relation_prompt_light % (
        args.width) + question + '\nTopic Entity: ' + entity_name + '\nRelations: ' + '; '.join(
        total_relations) + "\nThe output is: \n"


def clean_relations(string, entity_id, head_relations, valid_relations):
    # parse quoted strings and keep only valid relations (avoids eval on model output)
    candidates = re.findall(r"['\"]([^'\"]+)['\"]", string)
    relations = []
    seen = set()
    score = 0
    for relation in candidates:
        if relation in seen:
            continue
        seen.add(relation)
        if relation in head_relations:
            relations.append({"entity": entity_id, "relation": relation, "score": score, "head": True})
        elif relation in valid_relations:
            relations.append({"entity": entity_id, "relation": relation, "score": score, "head": False})
    if not relations:
        return False, "No relations found"
    return True, relations


def clean_entities(string, total_entities):
    pattern = r"{\s*(?P<entity>[^:]+)\s+\(Score:\s+(?P<score>[0-9.]+)\)}"
    entities = []
    for match in re.finditer(pattern, string):
        entity = match.group("entity").strip()
        if ';' in entity: continue
        score = match.group("score")
        if not entity or not score: return False, "output uncompleted.."
        try: score = float(score)
        except ValueError: return False, "Invalid score"
        entities.append({"entity": entity, "score": score})
    if not entities: return False, "No entities found"
    return True, entities


def summarize_retrival_results(total_candidates_id, pre_relations, raw_topic_entity, retrieve_entities, total_triplets):
    chain_of_entities = []
    entities_id = []
    topic_entities_name = list(raw_topic_entity.values())
    topic_entities_ids = list(raw_topic_entity.keys())
    for entity_select in retrieve_entities:
        if entity_select not in total_candidates_id: continue
        for temp_triple_select in total_triplets:
            if (entity_select.startswith('m.') and entity_select not in topic_entities_name and entity_select not in
                    topic_entities_ids and entity_select in temp_triple_select):
                chain_of_entities.append(temp_triple_select)
                pre_relation = temp_triple_select.split(', ')[1]
                if '|' in pre_relation:
                    pre_relations.extend([pre_relation.split('|')[0], pre_relation.split('|')[1]])
                else:
                    pre_relations.append(pre_relation)
                if "(specific value)" not in entity_select:
                    entities_id.append(entity_select.split('(')[0].strip())
    entities_id = [entity_id for entity_id in entities_id if entity_id.startswith('m.')]
    chain_of_entities = list(dict.fromkeys(chain_of_entities))
    return pre_relations, entities_id, chain_of_entities


def entity_search(entity, relation, value_entity_dict, id_name_dict, value_entity_list, head=True):
    if head:
        tail_entities_extract = sparql_tail_entities_extract % (entity, relation)
        entities = execurte_sparql(tail_entities_extract)
    else:
        head_entities_extract = sparql_head_entities_extract % (relation, entity)
        entities = execurte_sparql(head_entities_extract)
    if entities is not None and len(entities) > 0:
        entity_ids = replace_entities_prefix(entities)
        new_entity = [e for e in entity_ids if e.startswith("m.")]
    else:
        entity_ids = []
        new_entity = []
    if len(entity_ids) > 0 and len(new_entity) == 0:
        new_entity = []
        for entity_id in entity_ids:
            entity_key = simple_hash_encrypt(entity_id)
            entity_value = entity_id
            if entity_key not in value_entity_dict:
                value_entity_dict[entity_key] = entity_value
                id_name_dict[entity_key] = entity_value
            if entity_key not in value_entity_list:
                value_entity_list.append(entity_key)
            new_entity.append(entity_key)
    return new_entity, value_entity_dict, value_entity_list, id_name_dict


def entity_score_with_description(topic_entity_name, value_entity_dict, value_entity_list,
                                  raw_topic_entity, entity_relation, question, head_relation, tail_relation,
                                  id_description_dict, id_type_dict, id_name_dict, entity_candidates_id,
                                  call_num, args):
    relation = entity_relation["relation"]
    head = entity_relation["head"]
    topic_entity_id = entity_relation["entity"]
    entity_candidates_name = [id2entity_name_or_type(value_entity_dict, raw_topic_entity, entity_id) for entity_id in entity_candidates_id]
    current_t = {'total': 0, 'input': 0, 'output': 0}
    cvt_flag = False

    if all_unknown_entity(entity_candidates_name):
        next_relation_list = []
        for cvt_entity_id in entity_candidates_id:
            if head:
                sparql_relations_extract_head = sparql_head_relations % cvt_entity_id
                head_relations = execurte_sparql(sparql_relations_extract_head)
                head_relations = replace_relation_prefix(head_relations)
                if args.remove_unnecessary_rel:
                    head_relations = [r for r in head_relations if not abandon_rels(r)]
                head_relations = [r + '|' + next_relation for next_relation in head_relations]
                next_relation_list.extend(head_relations)
            else:
                sparql_relations_extract_tail = sparql_tail_relations % cvt_entity_id
                tail_relations = execurte_sparql(sparql_relations_extract_tail)
                tail_relations = replace_relation_prefix(tail_relations)
                if args.remove_unnecessary_rel:
                    tail_relations = [r for r in tail_relations if not abandon_rels(r)]
                tail_relations = [next_relation + '|' + r for next_relation in tail_relations]
                next_relation_list.extend(tail_relations)
        if len(next_relation_list) == 0: return [], [], [], current_t, call_num, cvt_flag
        next_relation_list = list(set(next_relation_list))
        flag = False
        retrieve_relations_with_scores = []

        relation_name_list = []
        relation_score_list = []
        if flag:
            relation_with_score_list = retrieve_relations_with_scores[:args.width]
            for relation_with_score in relation_with_score_list:
                relation_name_list.append(relation_with_score["relation"])
                relation_score_list.append(relation_with_score["score"])
        else:
            response_input = {'width': args.width, 'question': question, 'total_relations': '&;& '.join(next_relation_list)}
            response = requests.post(SLMPATH, json=response_input)
            relation_name_list = response.json()['topn_relations']
            relation_score_list = []
        next_relation_list = []
        for relation_name in relation_name_list:
            if '|' in relation_name:
                next_relation = relation_name.split('|')[1] if head else relation_name.split('|')[0]
                next_relation_list.append(next_relation)
        if len(next_relation_list) == 0: return [], [], [], current_t, call_num, cvt_flag

        final_entity_candidates_id, final_entity_candidates_name, final_topn_scores = [], [], []

        for relation_name in relation_name_list:
            if '|' not in relation_name: continue
            if head:
                next_relation = relation_name.split('|')[1]
                next_entity_candidates_id, value_entity_dict = entity_search_cvt(topic_entity_id, relation, next_relation, value_entity_dict)
            else:
                next_relation = relation_name.split('|')[0]
                next_entity_candidates_id, value_entity_dict = entity_search_cvt_tail(topic_entity_id, relation, next_relation, value_entity_dict)
            next_entity_candidates_id = list(set(next_entity_candidates_id) - set(topic_entity_id))
            if len(next_entity_candidates_id) >= 20:
                next_entity_candidates_id = random.sample(next_entity_candidates_id, 20)
            next_entity_candidates_name = [id2entity_name_or_type(value_entity_dict, raw_topic_entity, e) for e in next_entity_candidates_id]
            if not next_entity_candidates_name: continue
            if len(next_entity_candidates_name) > 50:
                response_input = {'width': 50, 'question': question, 'total_relations': '&;& '.join(next_entity_candidates_name)}
                response = requests.post(SLMPATH, json=response_input)
                topn_scores = response.json()['topn_scores']
                topn_names = response.json()['topn_relations']
            else:
                topn_scores = [1] * len(next_entity_candidates_name)
                topn_names = next_entity_candidates_name
            temp_entity_candidates_name, temp_entity_candidates_id = [], []
            final_topn_scores.extend(topn_scores)
            for index in range(len(topn_scores)):
                entity_index = next_entity_candidates_name.index(topn_names[index])
                temp_entity_candidates_id.append(next_entity_candidates_id[entity_index])
            final_entity_candidates_id.extend(temp_entity_candidates_id)
            if head: tail_relation = relation_name
            call_num += 1
            id_description_dict, id_type_dict, id_name_dict, token_num = entity_annotation_all(topic_entity_name, question, temp_entity_candidates_id, head_relation, tail_relation, id_description_dict, id_type_dict, id_name_dict, value_entity_dict, value_entity_list, args)
            for kk in token_num: current_t[kk] += token_num[kk]
            for entity_id in temp_entity_candidates_id:
                entity_type_plain = entity_id + (' (' + id_type_dict[entity_id] + ')' if entity_id in id_type_dict else ' (entity)')
                temp_entity_candidates_name.append(entity_type_plain)
            if head:
                temp_entity_candidates_name = ['|' + next_relation + ', ' + topn_name for topn_name in temp_entity_candidates_name]
            else:
                temp_entity_candidates_name = [topn_name + ', ' + next_relation + '|' for topn_name in temp_entity_candidates_name]
            final_entity_candidates_name.extend(temp_entity_candidates_name)
        return final_topn_scores, final_entity_candidates_name, final_entity_candidates_id, current_t, call_num, cvt_flag

    else:
        un_entity_index = [index for index, element in enumerate(entity_candidates_name) if element.startswith('UnName_Entity')]
        for i in un_entity_index: entity_candidates_id[i] = "UnName_Entity_ID"
        entity_candidates_id = [c for c in entity_candidates_id if c != "UnName_Entity_ID"]
        entity_candidates_name = [c for c in entity_candidates_name if not c.startswith('UnName_Entity')]
        final_entity_candidates_id, final_entity_candidates_name, topn_scores = [], [], []
        if len(entity_candidates_name) > 50:
            response_input = {'width': 50, 'question': question, 'total_relations': "&;& ".join(entity_candidates_name)}
            response = requests.post(SLMPATH, json=response_input)
            topn_scores = response.json()['topn_scores']
            topn_names = response.json()['topn_relations']
            for index in range(len(topn_names)):
                entity_index = entity_candidates_name.index(topn_names[index])
                final_entity_candidates_id.append(entity_candidates_id[entity_index])
        else:
            final_entity_candidates_id = entity_candidates_id
            topn_scores = [1] * len(entity_candidates_id)
        call_num += 1
        id_description_dict, id_type_dict, id_name_dict, token_num = entity_annotation_all(topic_entity_name, question, final_entity_candidates_id, head_relation, tail_relation, id_description_dict, id_type_dict, id_name_dict, value_entity_dict, value_entity_list, args)
        for kk in token_num: current_t[kk] += token_num[kk]
        for entity_id in final_entity_candidates_id:
            entity_type_plain = entity_id + (' (' + id_type_dict[entity_id] + ')' if entity_id in id_type_dict else ' (entity)')
            final_entity_candidates_name.append(entity_type_plain)
        return topn_scores, final_entity_candidates_name, final_entity_candidates_id, current_t, call_num, cvt_flag


def all_unknown_entity(entity_candidates): return all(c == "UnName_Entity" for c in entity_candidates)


def entity_search_cvt(entity, relation, next_relation, value_entity_dict):
    cvt_entities_extract = sparql_cvt_entities_extract % (entity, relation, next_relation)
    entities = execurte_sparql(cvt_entities_extract)
    entity_ids = replace_entities_prefix(entities)
    new_entity = [e for e in list(set(entity_ids)) if e.startswith("m.") or e.startswith("g.")]
    if len(entity_ids) > 0 and len(new_entity) == 0:
        for entity_id in entity_ids:
            entity_key = simple_hash_encrypt(entity_id)
            if entity_key not in value_entity_dict: value_entity_dict[entity_key] = entity_id
            new_entity.append(entity_key)
    return new_entity, value_entity_dict


def entity_search_cvt_tail(entity, relation, next_relation, value_entity_dict):
    cvt_entities_extract = sparql_cvt_entities_extract_tail % (relation, next_relation, entity)
    entities = execurte_sparql(cvt_entities_extract)
    entity_ids = replace_entities_prefix(entities)
    new_entity = [e for e in list(set(entity_ids)) if e.startswith("m.") or e.startswith("g.")]
    if len(entity_ids) > 0 and len(new_entity) == 0:
        for entity_id in entity_ids:
            entity_key = simple_hash_encrypt(entity_id)
            if entity_key not in value_entity_dict: value_entity_dict[entity_key] = entity_id
            new_entity.append(entity_key)
    return new_entity, value_entity_dict


def entity_annotation_all(topic_entity_name, question, entity_ids, head_relation, tail_relation, id_description_dict, id_type_dict, id_name_dict, value_entity_dict, value_entity_list, args):
    entity_type = ""
    entity_description = [""]
    token_num = {'total': 0, 'input': 0, 'output': 0}
    for entity_id in entity_ids:
        if entity_id in value_entity_list:
            entity_type = "specific value"
            entity_description = ["specific value"]
            break
    for entity_id in entity_ids:
        if entity_id not in id_name_dict:
            id_name_dict[entity_id] = id2entity_name_or_type(value_entity_dict, {}, entity_id)
    if entity_type != "specific value":
        if args.entity_abs:
            try:
                entity_description_prompt = ids_2_entities_description(tail_relation, head_relation, question, entity_ids)
                abs_entity_result, token_num = run_llm(entity_description_prompt, args.temperature_exploration, args.max_length, args.opeani_api_keys, args.LLM_type)
                restore_dict = eval(abs_entity_result.strip())
            except Exception as e:
                restore_dict = {'type': "entity", 'description': "sth. or sb."}
        else: restore_dict = {'type': "entity", 'description': "sth. or sb."}
        entity_type = restore_dict['type'].strip()
    for entity_id in entity_ids:
        if entity_id not in id_type_dict: id_type_dict[entity_id] = entity_type
        elif entity_type.lower() not in id_type_dict[entity_id].lower(): id_type_dict[entity_id] += ', ' + entity_type
        if entity_id in id_description_dict: entity_description = id_description_dict[entity_id]
        if tail_relation != '':
            entity_description.append(topic_entity_name + ', ' + tail_relation + ', ' + entity_id + ' (' + id_type_dict[entity_id] + ')')
        else:
            entity_description.append(entity_id + ' (' + id_type_dict[entity_id] + '), ' + head_relation + ', ' + topic_entity_name)
        id_description_dict[entity_id] = entity_description
    return id_description_dict, id_type_dict, id_name_dict, token_num


def update_history(topic_entity, entity_candidates, entity, scores, entity_candidates_id, total_candidates, total_scores, total_relations, total_entities_id, total_topic_entities, total_topic_entities_name, total_head):
    total_candidates.extend(entity_candidates)
    total_scores.extend(scores)
    total_relations.extend([entity['relation']] * len(entity_candidates))
    total_entities_id.extend(entity_candidates_id)
    total_topic_entities.extend([entity['entity']] * len(entity_candidates))
    total_topic_entities_name.extend([topic_entity[entity['entity']]] * len(entity_candidates))
    total_head.extend([entity['head']] * len(entity_candidates))
    return total_candidates, total_scores, total_relations, total_entities_id, total_topic_entities, total_topic_entities_name, total_head


def reasoning(question, cluster_chain_of_entities, id_name_dict, current_retrieve_depth, end_flag, call_num, args):
    current_t = {'total': 0, 'input': 0, 'output': 0}
    user_judge_prompt = judge_prompt_light + question + '\nKnowledge Triplets: ' + "".join(['\n'.join(t) + '\n' for t in cluster_chain_of_entities]) + 'A: '
    call_num += 1
    judge_response, token_num = run_llm(user_judge_prompt, args.temperature_reasoning, args.max_length, args.opeani_api_keys, args.LLM_type)
    for kk in token_num: current_t[kk] += token_num[kk]
    flag = extract_answer(judge_response)
    if if_true(flag) or current_retrieve_depth >= args.depth or end_flag:
        if args.filter_judge:
            chain_prompt = "".join(['\n'.join(t) + '\n' for t in cluster_chain_of_entities])
            if args.reason_with_name:
                for entity_id in id_name_dict: chain_prompt = chain_prompt.replace(entity_id, id_name_dict[entity_id])
            user_generator_prompt = generator_prompt + question + '\nKnowledge Triplets: ' + chain_prompt + 'A: '
            call_num += 1
            generator_response, token_num = run_llm(user_generator_prompt, args.temperature_reasoning, args.max_length, args.opeani_api_keys, args.LLM_type)
        else:
            call_num += 1
            generator_response, token_num = generate_without_explored_paths(question, args)
        for kk in token_num: current_t[kk] += token_num[kk]
        return True, (None if args.reason_with_name else generator_response), (generator_response if args.reason_with_name else None), current_t, call_num
    else:
        return False, judge_response, None, current_t, call_num


def extract_answer(text):
    start, end = text.find("{"), text.find("}")
    return text[start + 1:end].strip() if start != -1 and end != -1 else ""


def if_true(prompt): return prompt.lower().strip().replace(" ", "") == "yes"


def save_2_jsonl(task_id, value_entity_dict, raw_topic_entity, question, answer, cluster_chain_of_entities, id_name_dict, call_num, match_times, answer_list, token_num, tt, args):
    cluster_chain_str = str(cluster_chain_of_entities)
    for entity_id in id_name_dict:
        answer = answer.replace(entity_id, id_name_dict[entity_id])
        cluster_chain_str = cluster_chain_str.replace(entity_id, id_name_dict[entity_id])
    res = {"question": question, "results": answer, "reasoning_chains": cluster_chain_of_entities, "reasoning_chains_str": cluster_chain_str, "call_num": call_num, "match_times": match_times, "answer_list": answer_list, "token_num": token_num, "time": tt}
    with open(f"ARoG_0720_naive_1_{args.dataset}_{args.question_abs}_{args.entity_abs}_depth_{args.depth}_width_{args.width}_{task_id}.jsonl", "a") as f:
        f.write(json.dumps(res) + "\n")


def save_2_jsonl_with_name(task_id, value_entity_dict, raw_topic_entity, question, answer, cluster_chain_of_entities, id_name_dict, call_num, match_times, answer_list, token_num, tt, args):
    cluster_chain_str = str(cluster_chain_of_entities)
    for entity_id in id_name_dict:
        answer = answer.replace(entity_id, id_name_dict[entity_id])
        cluster_chain_str = cluster_chain_str.replace(entity_id, id_name_dict[entity_id])
    res = {"question": question, "results": answer, "reasoning_chains": cluster_chain_of_entities, "reasoning_chains_str": cluster_chain_str, "call_num": call_num, "match_times": match_times, "answer_list": answer_list, "token_num": token_num, "time": tt}
    with open(f"ARoG_0720_naive_1_with_name_{args.dataset}_{args.question_abs}_{args.entity_abs}_depth_{args.depth}_width_{args.width}_{task_id}.jsonl", "a") as f:
        f.write(json.dumps(res) + "\n")


def retrieve_top_docs(query, docs, model, width=3):
    query_emb, doc_emb = model.encode(query), model.encode(docs)
    scores = util.dot_score(query_emb, doc_emb)[0].cpu().tolist()
    doc_score_pairs = sorted(list(zip(docs, scores)), key=lambda x: x[1], reverse=True)
    return [p[0] for p in doc_score_pairs[:width]], [p[1] for p in doc_score_pairs[:width]]


def compute_bm25_similarity(query, corpus, width=3):
    tokenized_corpus = [doc.split(" ") for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = query.split(" ")
    return bm25.get_top_n(tokenized_query, corpus, n=width), sorted(bm25.get_scores(tokenized_query), reverse=True)[:width]


def clean_relations_bm25_sent(topn_relations, topn_scores, entity_id, head_relations):
    if all(s == 0 for s in topn_scores): topn_scores = [1/len(topn_scores)] * len(topn_scores)
    return True, [{"entity": entity_id, "relation": r, "score": topn_scores[i], "head": r in head_relations} for i, r in enumerate(topn_relations)]

In [ ]:
p = "/kaggle/working/ARoG/utils.py"
s = open(p, encoding="utf-8").read()
s = s.replace("head_relations = [r + '|' + next_relation for next_relation in head_relations]",
              "head_relations = [relation + '|' + next_relation for next_relation in head_relations]")
s = s.replace("tail_relations = [next_relation + '|' + r for next_relation in tail_relations]",
              "tail_relations = [next_relation + '|' + relation for next_relation in tail_relations]")
open(p, "w", encoding="utf-8").write(s)

In [ ]:
%cd /kaggle/working/ARoG
!sed -i 's#^            print(sparql_query)#            pass  \# print(sparql_query)#' freebase_func.py
!grep -n "print(sparql_query)" freebase_func.py

In [ ]:
%cd /kaggle/working/ARoG
import os, shutil, subprocess, re
if not os.path.exists("data/WebQSP_original.json"):
    shutil.copy("data/WebQSP.json", "data/WebQSP_original.json")
print(subprocess.run(["python", "/kaggle/working/aciids_fv/protoA/patch_arog.py"], capture_output=True, text=True).stdout)
src = open("main_freebase.py", encoding="utf-8").read()
active = re.findall(r"(?m)^\s*datas = random\.sample\(datas, int\(len\(datas\) / \d+\)\)", src)
assert not active, f"main_freebase.py still subsamples: {active}"
print("patched; no subsampling")

## 4. Run ARoG in three chunks

Each chunk holds 40 patients x 3 levels, balanced by class. If the session stops, rerun sections 1 to 3 and then only the unfinished chunks; a chunk whose log is complete is skipped.

In [ ]:
import pandas as pd, json, os, time, subprocess, collections
rec = pd.read_csv(f"{WORK}/testA/records_PRIVATE.csv")
Q = {q["QuestionId"]: q for q in json.load(open(f"{WORK}/testA/WebQSP.json"))}
pat = rec.drop_duplicates("patient_mid").sort_values(["class_key", "patient_mid"]).copy()
pat["chunk"] = pat.groupby("class_key").cumcount() % 3
rec = rec.merge(pat[["patient_mid", "chunk"]], on="patient_mid")
CHUNKS = {k: rec[rec.chunk == k].record_id.tolist() for k in range(3)}
print({k: len(v) for k, v in CHUNKS.items()}, "| patients per class:", rec.drop_duplicates("patient_mid").class_key.value_counts().to_dict())
assert len(rec) == 360 and sum(len(v) for v in CHUNKS.values()) == 360

def run_chunk(k):
    os.chdir(f"{WORK}/ARoG")
    logf = f"{WORK}/ARoG/arog_test_chunk{k}.jsonl"; ids = CHUNKS[k]
    if os.path.exists(logf):
        have = {json.loads(l)["qid"] for l in open(logf)}
        if set(ids) <= have:
            print(f"chunk {k}: log complete, skipped"); return
        os.remove(logf)
    json.dump([Q[i] for i in ids], open("data/WebQSP.json", "w"))
    env = dict(os.environ, AROG_LOG=logf); t0 = time.time()
    r = subprocess.run(["arog_env/bin/python", "main_freebase.py", "--dataset", "webqsp", "--width", "3", "--depth", "3",
                        "--opeani_api_keys", "none"], env=env, capture_output=True, text=True)
    have = {json.loads(l)["qid"] for l in open(logf)}
    print(f"chunk {k}: {time.time()-t0:.0f}s returncode={r.returncode} records logged: {len(have & set(ids))}/{len(ids)}")
    if r.returncode != 0: print(r.stderr[-1500:])
    assert r.returncode == 0 and set(ids) <= have, f"chunk {k} incomplete"
print("ready")

In [ ]:
if 0 in RUN_ARoG_CHUNKS: run_chunk(0)

In [ ]:
if 1 in RUN_ARoG_CHUNKS: run_chunk(1)

In [ ]:
if 2 in RUN_ARoG_CHUNKS: run_chunk(2)

In [ ]:
# merge chunk logs and summarise
os.chdir(WORK)
with open(f"{WORK}/arog_transcripts_test.jsonl", "w") as fo:
    for k in range(3):
        fo.write(open(f"{WORK}/ARoG/arog_test_chunk{k}.jsonl").read())
calls = collections.defaultdict(list)
for l in open(f"{WORK}/arog_transcripts_test.jsonl"):
    d = json.loads(l); calls[d["qid"]].append(d)
R_ = rec.set_index("record_id")
D = pd.DataFrame([dict(level=R_.at[r, "level"], n_calls=len(cs), chars=sum(len(c["prompt"]) for c in cs),
                       has_rel=any("clinical." in c["prompt"] for c in cs), empty=sum(1 for c in cs if not c["response"].strip()))
                  for r, cs in calls.items()])
print(len(D), "records with transcripts / 360")
print(D.groupby("level").agg(n=("n_calls","size"), calls_med=("n_calls","median"), chars_max=("chars","max"),
      over40k=("chars", lambda x: (x > 40000).sum()), empty_resp=("empty","sum"), no_rel=("has_rel", lambda x: (~x).sum())))
assert len(D) == 360

## 5. Primary analysis: A3 on relation labels from the real ARoG prompts

In [ ]:
%cd /kaggle/working/aciids_fv
!python build_views.py --base /kaggle/working/ACIIDS_2027 --split train,validation --out out/train_views.jsonl --labels out/train_labels_PRIVATE.csv
!python build_views.py --base /kaggle/working/ACIIDS_2027 --split test --prereg $PREREG --out out/test_views.jsonl --labels out/test_labels_PRIVATE.csv
%cd protoA
!python stat_on_transcripts.py --prereg $PREREG --transcripts /kaggle/working/arog_transcripts_test.jsonl --records /kaggle/working/testA/records_PRIVATE.csv --train_views ../out/train_views.jsonl --train_labels ../out/train_labels_PRIVATE.csv --dev_views ../out/test_views.jsonl --out /kaggle/working/testA/stat_transcripts.csv

In [ ]:
# intermediate archive in case the session stops
import shutil, os
os.makedirs("/kaggle/working/outputs_test", exist_ok=True)
for f in [PREREG, "/kaggle/working/arog_transcripts_test.jsonl", "/kaggle/working/testA/stat_transcripts.csv",
          "/kaggle/working/testA/records_PRIVATE.csv", "/kaggle/working/testA/WebQSP.json"] + [f"/kaggle/working/ARoG/arog_test_chunk{k}.jsonl" for k in range(3)]:
    if os.path.exists(f): shutil.copy(f, "/kaggle/working/outputs_test/")
shutil.make_archive("/kaggle/working/protocolA_test_outputs", "zip", "/kaggle/working/outputs_test")
print("written /kaggle/working/protocolA_test_outputs.zip")

## 6. Secondary analyses: zero-shot LLM attacker (A1)

In [ ]:
%cd /kaggle/working/aciids_fv/protoA
RUNP = "--base /kaggle/working/ACIIDS_2027 --transcripts /kaggle/working/arog_transcripts_test.jsonl --data /kaggle/working/testA/WebQSP.json --records /kaggle/working/testA/records_PRIVATE.csv --backend ollama --model qwen2.5:7b-instruct --prereg /kaggle/working/PREREG_LOCK.json"
!python run_transcript_attack.py $RUNP --out /kaggle/working/testA/pred_xtract.jsonl --jobs xtract,xtract_c2tok,c0
%cd /kaggle/working/aciids_fv
!python evaluate.py --prereg $PREREG --pred /kaggle/working/testA/pred_xtract.jsonl --labels /kaggle/working/testA/records_PRIVATE.csv --out /kaggle/working/testA/metrics_xtract.csv --job xtract

In [ ]:
if RUN_A1_RAW:
    %cd /kaggle/working/aciids_fv/protoA
    !python run_transcript_attack.py $RUNP --out /kaggle/working/testA/pred_raw.jsonl --jobs transcript,c0,c2tok
    %cd /kaggle/working/aciids_fv
    !python evaluate.py --prereg $PREREG --pred /kaggle/working/testA/pred_raw.jsonl --labels /kaggle/working/testA/records_PRIVATE.csv --out /kaggle/working/testA/metrics_raw.csv --job transcript

In [ ]:
if RUN_ABLATION:
    %cd /kaggle/working/aciids_fv/protoA
    !python run_transcript_attack.py $RUNP --stages relation_search_prune --out /kaggle/working/testA/pred_abl_prune.jsonl --jobs transcript,c0
    !python run_transcript_attack.py $RUNP --stages relation_search_prune,entity_annotation_all,reasoning --out /kaggle/working/testA/pred_abl_post.jsonl --jobs transcript,c0
    %cd /kaggle/working/aciids_fv
    for n in ("prune", "post"):
        !python evaluate.py --prereg $PREREG --pred /kaggle/working/testA/pred_abl_$n.jsonl --labels /kaggle/working/testA/records_PRIVATE.csv --out /kaggle/working/testA/metrics_abl_$n.csv --job transcript

In [ ]:
if RUN_A1_FIXED:   # same 120 patients (record_id is shared by both tools)
    import json
    ids = set(rec.record_id)
    V = [l for l in open("/kaggle/working/aciids_fv/out/test_views.jsonl") if json.loads(l)["record_id"] in ids]
    open("/kaggle/working/testA/views_sample.jsonl", "w").writelines(V); print("views:", len(V))
    %cd /kaggle/working/aciids_fv
    !python run_attack.py --base /kaggle/working/ACIIDS_2027 --views /kaggle/working/testA/views_sample.jsonl --out /kaggle/working/testA/pred_fixed.jsonl --backend ollama --model $OLLAMA_TAG --jobs triples,c0,c2tok
    !python evaluate.py --prereg $PREREG --pred /kaggle/working/testA/pred_fixed.jsonl --labels /kaggle/working/testA/records_PRIVATE.csv --out /kaggle/working/testA/metrics_fixed.csv

## 7. Secondary: A3 on fixed views of all 240 test patients (not ARoG traffic), and final packaging

In [ ]:
%cd /kaggle/working/aciids_fv
!python stat_attackers.py --base /kaggle/working/ACIIDS_2027 --prereg $PREREG --train_views out/train_views.jsonl --train_labels out/train_labels_PRIVATE.csv --eval_views out/test_views.jsonl --eval_labels out/test_labels_PRIVATE.csv --out /kaggle/working/testA/stat_fixed_all240.csv

In [ ]:
import shutil, os, glob
os.makedirs("/kaggle/working/outputs_test", exist_ok=True)
for f in glob.glob("/kaggle/working/testA/*.csv") + glob.glob("/kaggle/working/testA/pred_*.jsonl") + [PREREG, "/kaggle/working/arog_transcripts_test.jsonl"]:
    shutil.copy(f, "/kaggle/working/outputs_test/")
shutil.make_archive("/kaggle/working/protocolA_test_outputs", "zip", "/kaggle/working/outputs_test")
print("written /kaggle/working/protocolA_test_outputs.zip; records_PRIVATE.csv holds the labels and stays private")

In [ ]:
import os, sys, json, hashlib, subprocess, platform, datetime
os.chdir("/kaggle/working")

# environment
env = {
    "time_utc": datetime.datetime.utcnow().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pip_freeze": subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                 capture_output=True, text=True).stdout.splitlines(),
}
try:
    env["ollama"] = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
except Exception as e:
    env["ollama"] = str(e)
json.dump(env, open("ENV.json", "w"), indent=2)

# checksums of result files (files over 200 MB are model weights or caches)
SKIP = (".ollama", "__pycache__", ".cache")
rows = []
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if not d.startswith(SKIP)]
    for fn in files:
        p = os.path.join(root, fn)
        s = os.path.getsize(p)
        if s > 200 * 1024**2:
            print("skipped (large):", p, s // 1024**2, "MB"); continue
        h = hashlib.sha256(open(p, "rb").read()).hexdigest()
        rows.append(f"{h}  {s:>12}  {p}")
open("SHA256SUMS.txt", "w").write("\n".join(sorted(rows, key=lambda r: r.split()[-1])))
print(len(rows), "files hashed")

# archive
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zipname = f"results_{stamp}.zip"
!zip -r -q {zipname} . -x "*.zip" "*/.ollama/*" "*/__pycache__/*" "*/.cache/*" -x@<(find . -size +200M)
!ls -lh {zipname}

from IPython.display import FileLink
FileLink(zipname)